# Data Platform Foundations: Reliability Mechanics for Production Pipelines

Maps to `design3.md` Phase 3.

This notebook isolates the reliability mechanics that turn pipelines from fragile scripts into platform components. Every section follows the same pattern: why it matters, how it works, industry context, working code, and a prompt to extend.

These are not theoretical concepts. They are the mechanics that determine whether your pipeline silently corrupts data at 3 AM or recovers cleanly and tells you what happened.

Goal: build working fluency with retries, idempotency, data contracts, quality gates, dead letter queues, lineage, CDC, batch vs stream tradeoffs, and observability so that you can design and operate reliable data systems without guessing.

## Learning Goals

By the end of this notebook, you should be able to:

- implement retry logic with exponential backoff and jitter, and explain why each piece matters
- design idempotent pipeline operations and explain the difference between exactly-once and at-least-once
- define data contracts and handle schema evolution without breaking downstream consumers
- build multi-stage data quality gates that distinguish warnings from blocking failures
- implement dead letter queue and quarantine patterns for bad records
- explain data lineage and design safe backfill operations
- describe CDC approaches and when each is appropriate
- make informed batch vs stream architecture decisions based on actual tradeoffs
- define meaningful SLIs, SLOs, and operational metrics for data pipelines

Working rule:

- read the explanation first
- run the code cell
- modify one example before moving on
- write one short note in your own words after each section

## 1. Retries, Backoff, and Timeouts

Transient failures are normal in distributed systems. Network blips, temporary overloads, lock contention, DNS hiccups, and brief service restarts all happen regularly. The question is not whether your pipeline will encounter failures, but how it behaves when it does.

### Why retries matter

A naive pipeline that fails on the first transient error and stops is brittle. Most transient errors resolve within seconds. A pipeline that retries intelligently can ride through brief disruptions without human intervention, and that is the difference between a system that pages you at 3 AM and one that self-heals.

### Retry strategies

**Fixed interval**: wait the same amount of time between each retry (e.g., 2 seconds every time). Simple, but creates synchronized retry storms when many clients fail at the same time.

**Exponential backoff**: double the wait time on each retry (e.g., 1s, 2s, 4s, 8s). This gives the failing service progressively more breathing room to recover, which is critical when the failure is caused by overload.

**Exponential backoff with jitter**: add randomness to the backoff delay. Without jitter, if 100 clients all fail at the same moment and all use the same exponential backoff, they will all retry at the same moments (1s, 2s, 4s...), creating a synchronized thundering herd that hammers the recovering service. Jitter spreads the retries across time, which is one of the most important details in production retry logic.

### Timeouts: connect vs read

You must always set both a connect timeout and a read timeout:

- **Connect timeout**: how long to wait to establish a TCP connection. If a service is down or unreachable, you want to fail fast rather than waiting the OS default (often 60-120 seconds).
- **Read timeout**: how long to wait for the server to send a response after the connection is established. A server that accepts connections but hangs on processing will hold your thread hostage without a read timeout.

Setting neither timeout means your pipeline can hang indefinitely on a single stuck request, blocking everything downstream.

### Circuit breaker pattern

Retries are good for transient failures, but what about sustained failures? If a service is clearly down, continuing to retry wastes resources and can make things worse. The circuit breaker pattern solves this:

- **Closed** (normal): requests flow through. Track failure rate.
- **Open** (tripped): too many recent failures. Reject requests immediately without calling the service. This is "fail fast."
- **Half-open** (testing): after a cooldown period, allow one test request through. If it succeeds, close the circuit. If it fails, stay open.

This pattern prevents a failing downstream service from cascading failures back through your entire pipeline.

### When NOT to retry

- **Non-idempotent operations without idempotency keys**: if the operation might have succeeded but you did not get the acknowledgment, retrying could create duplicates (e.g., charging a credit card twice). We cover idempotency in the next section.
- **4xx client errors**: these indicate a bug in your request (bad payload, missing auth, invalid parameters). Retrying will not fix them. Only retry on 5xx (server errors) and network-level failures.
- **Poison pill messages**: a malformed record that crashes the consumer will crash it again on retry. These belong in a dead letter queue, not a retry loop.

In [ ]:
import random
import time
import functools
from typing import Any, Callable


# --- Retry with exponential backoff + jitter ---

def retry_with_backoff(
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 30.0,
    retryable_exceptions: tuple = (Exception,),
):
    """Decorator that retries a function with exponential backoff and jitter."""

    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            attempts = []
            for attempt in range(max_retries + 1):
                try:
                    result = func(*args, **kwargs)
                    attempts.append({"attempt": attempt + 1, "status": "success"})
                    return result, attempts
                except retryable_exceptions as exc:
                    if attempt == max_retries:
                        attempts.append({"attempt": attempt + 1, "status": "failed_final", "error": str(exc)})
                        raise
                    # Exponential backoff: base_delay * 2^attempt
                    backoff = min(base_delay * (2 ** attempt), max_delay)
                    # Full jitter: uniform random between 0 and backoff
                    jitter_delay = random.uniform(0, backoff)
                    attempts.append({
                        "attempt": attempt + 1,
                        "status": "retrying",
                        "error": str(exc),
                        "backoff_base": round(backoff, 3),
                        "actual_delay": round(jitter_delay, 3),
                    })
                    time.sleep(jitter_delay)
        return wrapper
    return decorator


# --- Simulate a flaky service ---

call_count = 0

@retry_with_backoff(max_retries=4, base_delay=0.1, max_delay=2.0)
def flaky_api_call(payload: str) -> dict:
    """Simulates a service that fails the first 2 calls, then succeeds."""
    global call_count
    call_count += 1
    if call_count <= 2:
        raise ConnectionError(f"Service unavailable (attempt {call_count})")
    return {"status": "ok", "payload": payload}


call_count = 0  # reset
result, attempt_log = flaky_api_call("trade_batch_42")

print("Attempt log:")
for entry in attempt_log:
    print(f"  {entry}")
print(f"\nFinal result: {result}")


# --- Compare backoff strategies visually ---

print("\n--- Backoff delay comparison (no actual sleep) ---")
print(f"{'Attempt':<10} {'Fixed(2s)':<12} {'Exponential':<14} {'Exp+Jitter':<14}")
random.seed(42)
for i in range(6):
    fixed = 2.0
    exponential = min(1.0 * (2 ** i), 30.0)
    jittered = random.uniform(0, exponential)
    print(f"{i+1:<10} {fixed:<12.3f} {exponential:<14.3f} {jittered:<14.3f}")

# Try next:
# 1. Change the flaky service to fail 4 times and watch the decorator exhaust retries.
# 2. Add a check that skips retry on ValueError (a "client error" analog).
# 3. Plot the three backoff strategies for 10 attempts to see the divergence.

In [ ]:
# --- Simple Circuit Breaker ---

class CircuitBreaker:
    """
    Minimal circuit breaker implementation.

    States:
      - CLOSED: requests pass through, failures are counted
      - OPEN: requests are rejected immediately (fail fast)
      - HALF_OPEN: one test request allowed through
    """

    def __init__(self, failure_threshold: int = 3, recovery_timeout: float = 5.0):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failure_count = 0
        self.state = "CLOSED"
        self.last_failure_time: float | None = None
        self.log: list[dict] = []

    def call(self, func: Callable, *args: Any, **kwargs: Any) -> Any:
        if self.state == "OPEN":
            elapsed = time.time() - (self.last_failure_time or 0)
            if elapsed >= self.recovery_timeout:
                self.state = "HALF_OPEN"
                self.log.append({"event": "half_open", "elapsed": round(elapsed, 2)})
            else:
                self.log.append({"event": "rejected", "state": "OPEN"})
                raise RuntimeError(f"Circuit is OPEN. Failing fast. Try again in {self.recovery_timeout - elapsed:.1f}s")

        try:
            result = func(*args, **kwargs)
            if self.state == "HALF_OPEN":
                self.state = "CLOSED"
                self.failure_count = 0
                self.log.append({"event": "recovered", "state": "CLOSED"})
            return result
        except Exception as exc:
            self.failure_count += 1
            self.last_failure_time = time.time()
            if self.failure_count >= self.failure_threshold:
                self.state = "OPEN"
                self.log.append({"event": "tripped", "state": "OPEN", "failures": self.failure_count})
            else:
                self.log.append({"event": "failure", "count": self.failure_count, "error": str(exc)})
            raise


# --- Demonstrate circuit breaker behavior ---

breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=1.0)
service_healthy = False

def unreliable_service() -> str:
    if not service_healthy:
        raise ConnectionError("Service down")
    return "OK"


# Phase 1: accumulate failures until circuit trips
for i in range(5):
    try:
        breaker.call(unreliable_service)
    except (ConnectionError, RuntimeError) as e:
        pass  # expected

print("Phase 1 - Service down, circuit should trip:")
for entry in breaker.log:
    print(f"  {entry}")

# Phase 2: service recovers, wait for half-open
service_healthy = True
time.sleep(1.1)  # wait past recovery_timeout

try:
    result = breaker.call(unreliable_service)
    print(f"\nPhase 2 - After recovery: {result}")
except RuntimeError as e:
    print(f"\nPhase 2 - Still blocked: {e}")

print(f"Final state: {breaker.state}")
print("\nFull log:")
for entry in breaker.log:
    print(f"  {entry}")

# Try next:
# 1. Set recovery_timeout to 10s and observe the OPEN rejection behavior.
# 2. Make the half-open test request fail and verify the circuit stays OPEN.
# 3. Add a success_count threshold for HALF_OPEN -> CLOSED (require N successes, not just 1).

## 2. Idempotency and Rerun Safety

Idempotency means that running the same operation twice produces the same result as running it once. This is one of the most important properties in data engineering because pipelines crash and restart, messages get replayed, operators hit "run" again, and schedulers sometimes double-fire.

### Why it matters

Without idempotency, every retry, restart, or rerun risks creating duplicate records. In a trading data pipeline, duplicate trade records corrupt VWAP calculations, inflate volume metrics, and produce wrong P&L numbers. In billing, duplicates charge customers twice. In analytics, duplicates skew every aggregate.

The insidious part is that non-idempotent pipelines often work fine during normal operation. The bugs only appear during failure recovery, which is exactly when you need reliability most.

### Idempotency key design

The idempotency key is a composite natural key that uniquely identifies a logical operation:

- **Source**: which system produced this record (e.g., "finnhub", "entsoe")
- **Business key**: the natural identifier within that source (e.g., trade ID, order ID)
- **Event time**: the timestamp of the event, not the processing time

The composite `source + business_key + event_time` ensures that replaying the same event from the same source produces the same key and therefore the same dedup behavior.

### Implementation patterns

**Upsert (INSERT ON CONFLICT UPDATE)**: the database handles idempotency. If the record exists, update it; if not, insert it. This is the simplest pattern and works well for row-level operations.

```sql
INSERT INTO trades (trade_id, symbol, price, volume, ts)
VALUES (:trade_id, :symbol, :price, :volume, :ts)
ON CONFLICT (trade_id) DO UPDATE SET price = EXCLUDED.price, volume = EXCLUDED.volume;
```

**Dedup table**: before processing a record, check if its idempotency key has already been processed. If yes, skip it. This works for any processing pattern, not just database inserts.

**Atomic rename (write-to-temp-then-rename)**: write output to a temporary file, then atomically rename it to the final location. If the pipeline crashes mid-write, the temp file is incomplete and the final location is untouched. On rerun, the temp file is overwritten cleanly.

### Exactly-once vs at-least-once

True exactly-once delivery is extremely difficult to achieve in distributed systems. What most production systems actually implement is **at-least-once delivery + idempotent consumer**. The message might be delivered more than once, but the consumer is designed so that processing it again has no additional effect.

This is why idempotency is a prerequisite for reliable retry behavior. Retries give you at-least-once. Idempotency turns at-least-once into effectively exactly-once.

In [ ]:
from datetime import datetime, timezone
from dataclasses import dataclass, field


# --- Idempotency key construction ---

def make_idempotency_key(source: str, business_key: str, event_time: str) -> str:
    """Build a composite idempotency key from source, business key, and event time."""
    return f"{source}|{business_key}|{event_time}"


# --- Dedup-based idempotent ingestion ---

class IdempotentIngester:
    """
    Simulates idempotent ingestion using a seen-keys set.
    In production, this would be a database table or Redis set.
    """

    def __init__(self):
        self.seen_keys: set[str] = set()
        self.stored_records: list[dict] = []
        self.stats = {"ingested": 0, "duplicates_skipped": 0}

    def ingest(self, record: dict) -> str:
        key = make_idempotency_key(
            record["source"], record["trade_id"], record["event_time"]
        )
        if key in self.seen_keys:
            self.stats["duplicates_skipped"] += 1
            return f"SKIP (duplicate): {key}"
        self.seen_keys.add(key)
        self.stored_records.append(record)
        self.stats["ingested"] += 1
        return f"INGESTED: {key}"


# --- Demonstrate: first run ---

trades = [
    {"source": "finnhub", "trade_id": "T001", "symbol": "EURUSD", "price": 1.0845, "volume": 1000, "event_time": "2026-04-04T08:15:00Z"},
    {"source": "finnhub", "trade_id": "T002", "symbol": "USDJPY", "price": 145.12, "volume": 500, "event_time": "2026-04-04T08:15:01Z"},
    {"source": "finnhub", "trade_id": "T003", "symbol": "GBPUSD", "price": 1.2641, "volume": 750, "event_time": "2026-04-04T08:15:02Z"},
]

ingester = IdempotentIngester()

print("=== First run (fresh ingestion) ===")
for trade in trades:
    print(f"  {ingester.ingest(trade)}")
print(f"  Stats: {ingester.stats}")
print(f"  Records stored: {len(ingester.stored_records)}")

# --- Demonstrate: simulated replay (same records arrive again) ---

print("\n=== Simulated replay (same records re-delivered) ===")
for trade in trades:
    print(f"  {ingester.ingest(trade)}")
print(f"  Stats: {ingester.stats}")
print(f"  Records stored: {len(ingester.stored_records)}")

# --- Demonstrate: what breaks WITHOUT idempotency ---

print("\n=== Without idempotency (naive append) ===")
naive_store: list[dict] = []
for trade in trades:
    naive_store.append(trade)
# Simulate replay
for trade in trades:
    naive_store.append(trade)
print(f"  Records stored: {len(naive_store)} (expected 3, got 6 -- duplicates!)")

# Try next:
# 1. Add a new trade with the same trade_id but different price. Should the ingester accept or reject it?
# 2. Implement an "upsert" mode where duplicates update the existing record instead of skipping.
# 3. Replace the in-memory set with a dictionary that also stores the ingestion timestamp.

## 3. Data Contracts and Schema Evolution

A data contract is an explicit agreement about what data looks like: its schema, its semantics, its quality guarantees, and its delivery cadence. Without contracts, upstream changes silently break downstream consumers, and the first sign of trouble is a wrong dashboard number or a failed model training run days later.

### Why data contracts matter

In a typical organization, one team produces data and many teams consume it. Without a contract, the producer can:

- add, remove, or rename fields at any time
- change the meaning of a field (e.g., "volume" shifts from shares to lots)
- change the precision of numeric fields
- change null semantics or default values
- change delivery timing

Each of these changes can silently corrupt every downstream consumer. Data contracts make these changes explicit, versioned, and negotiated.

### Schema evolution strategies

**Additive only (backward compatible)**: the strictest and safest rule. New fields are always optional with defaults. Existing fields are never removed or renamed. Old consumers ignore new fields. New consumers handle missing optional fields gracefully. This is the default strategy for Avro schemas in Kafka with Confluent Schema Registry.

**Versioned schemas**: when breaking changes are necessary, create an explicit new version (v1, v2) with a documented migration path. Both versions may coexist during a transition period. This is more complex but sometimes unavoidable.

**Schema registry**: a centralized service that stores and validates schemas. Producers register their schema before publishing. Consumers fetch the schema to deserialize. The registry enforces compatibility rules (backward, forward, full). Confluent Schema Registry is the standard for Kafka ecosystems.

### Breaking vs non-breaking changes

**Safe (non-breaking)**:
- Add an optional field with a default value
- Add a new enum value (if consumers handle unknown values gracefully)
- Widen a numeric type (int32 to int64)
- Add a new topic or table

**Dangerous (breaking)**:
- Remove a field that consumers depend on
- Rename a field (looks like a remove + add to consumers)
- Change a field's type (string to int, float to decimal)
- Change the meaning of an existing field
- Make an optional field required
- Change the partitioning key

### Industry context

In the energy trading platform, the `TradeEvent` model has specific fields: UUID trade_id, symbol, price (Decimal 18,8), volume, side, timestamp. If a data provider starts sending volume in a different unit, or adds a new field, or changes the timestamp format, every downstream consumer (VWAP aggregation, storage, API) could break. A data contract catches this at the boundary.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal
from enum import Enum
from typing import Any


# --- Data contract: define what a trade event must look like ---

class TradeSide(str, Enum):
    BUY = "BUY"
    SELL = "SELL"


@dataclass(frozen=True)
class TradeContractV1:
    """
    Version 1 of the trade event data contract.
    This defines the required fields, types, and constraints.
    """
    trade_id: str          # UUID format
    symbol: str            # uppercase, e.g. "EURUSD"
    price: Decimal         # 18,8 precision, must be positive
    volume: int            # must be positive
    side: TradeSide        # BUY or SELL
    event_time: str        # ISO 8601 with timezone

    SCHEMA_VERSION = "1.0"


@dataclass(frozen=True)
class TradeContractV2:
    """
    Version 2: additive change -- new optional field 'venue'.
    This is backward compatible: old consumers can ignore 'venue'.
    """
    trade_id: str
    symbol: str
    price: Decimal
    volume: int
    side: TradeSide
    event_time: str
    venue: str | None = None  # NEW optional field with default

    SCHEMA_VERSION = "2.0"


# --- Schema validation ---

def validate_against_contract(record: dict, version: str = "1.0") -> dict:
    """
    Validate a raw record against the trade contract.
    Returns {"valid": bool, "errors": list[str], "warnings": list[str]}.
    """
    errors: list[str] = []
    warnings: list[str] = []

    # Required fields for v1
    required_v1 = {"trade_id", "symbol", "price", "volume", "side", "event_time"}
    missing = required_v1 - set(record.keys())
    if missing:
        errors.append(f"Missing required fields: {missing}")

    # Type and value checks
    if "price" in record:
        try:
            price = Decimal(str(record["price"]))
            if price <= 0:
                errors.append(f"Price must be positive, got {price}")
        except Exception:
            errors.append(f"Price is not a valid decimal: {record['price']}")

    if "volume" in record:
        if not isinstance(record["volume"], int) or record["volume"] <= 0:
            errors.append(f"Volume must be a positive integer, got {record['volume']}")

    if "side" in record:
        if record["side"] not in ("BUY", "SELL"):
            errors.append(f"Side must be BUY or SELL, got {record['side']}")

    # v2 awareness: detect unknown fields (schema drift detection)
    known_fields = required_v1 | {"venue", "source"}
    unknown = set(record.keys()) - known_fields
    if unknown:
        warnings.append(f"Unknown fields detected (possible schema drift): {unknown}")

    return {"valid": len(errors) == 0, "errors": errors, "warnings": warnings}


# --- Demonstrate contract validation ---

good_record = {
    "trade_id": "550e8400-e29b-41d4-a716-446655440000",
    "symbol": "EURUSD",
    "price": "1.08450000",
    "volume": 1000,
    "side": "BUY",
    "event_time": "2026-04-04T08:15:00+00:00",
}

bad_record = {
    "trade_id": "550e8400-e29b-41d4-a716-446655440001",
    "symbol": "EURUSD",
    "price": "-5.0",
    "volume": 0,
    "side": "HOLD",  # invalid
    # missing event_time
}

evolved_record = {
    "trade_id": "550e8400-e29b-41d4-a716-446655440002",
    "symbol": "USDJPY",
    "price": "145.12000000",
    "volume": 500,
    "side": "SELL",
    "event_time": "2026-04-04T08:15:01+00:00",
    "venue": "ECN",              # v2 field (safe additive change)
    "liquidity_pool": "DARK",    # unknown field (schema drift!)
}

print("=== Good record ===")
result = validate_against_contract(good_record)
print(f"  Valid: {result['valid']}, Errors: {result['errors']}, Warnings: {result['warnings']}")

print("\n=== Bad record (multiple violations) ===")
result = validate_against_contract(bad_record)
print(f"  Valid: {result['valid']}")
for err in result["errors"]:
    print(f"  ERROR: {err}")

print("\n=== Evolved record (v2 field + unknown field) ===")
result = validate_against_contract(evolved_record)
print(f"  Valid: {result['valid']}")
for w in result["warnings"]:
    print(f"  WARNING: {w}")

# --- Show backward-compatible schema evolution ---

print("\n=== Schema evolution: v1 -> v2 ===")
v1_trade = TradeContractV1(
    trade_id="T001", symbol="EURUSD", price=Decimal("1.0845"),
    volume=1000, side=TradeSide.BUY, event_time="2026-04-04T08:15:00Z"
)
v2_trade = TradeContractV2(
    trade_id="T002", symbol="USDJPY", price=Decimal("145.12"),
    volume=500, side=TradeSide.SELL, event_time="2026-04-04T08:15:01Z",
    venue="ECN"
)
v2_trade_no_venue = TradeContractV2(
    trade_id="T003", symbol="GBPUSD", price=Decimal("1.2641"),
    volume=750, side=TradeSide.BUY, event_time="2026-04-04T08:15:02Z"
    # venue defaults to None -- backward compatible
)

print(f"  V1 trade: {v1_trade}")
print(f"  V2 trade with venue: {v2_trade}")
print(f"  V2 trade without venue: {v2_trade_no_venue}")

# Try next:
# 1. Add a v3 contract with a "currency_pair" field that replaces "symbol". Is this breaking or safe?
# 2. Implement a schema migration function that converts v1 records to v2 format.
# 3. Add format validation for trade_id (must be UUID) and event_time (must be ISO 8601).

## 4. Data Quality Gates

A data quality gate is a checkpoint in your pipeline that inspects data before it moves to the next stage. The key insight is that quality gates form a pipeline themselves: schema check, then completeness check, then reasonableness check, then cross-validation. Each stage catches a different class of problem, and failing early avoids wasting compute on data that was broken from the start.

### The validation pipeline

**Stage 1 -- Schema check**: are the required fields present? Are types correct? Are formats valid? This catches structural problems: missing columns, wrong data types, malformed timestamps. If the schema is wrong, nothing downstream can work, so this must be first.

**Stage 2 -- Completeness check**: do we have the expected number of records? Are all expected symbols/sources present? Are there unexpected nulls in required fields? This catches data loss: a source that stopped sending, a partition that was skipped, a filter that was too aggressive.

**Stage 3 -- Reasonableness check**: are prices within expected ranges? Are timestamps not in the future? Are volumes positive and within plausible bounds? This catches semantic corruption: a price that is off by a factor of 1000 because the unit changed, a timestamp that wrapped around, a negative volume from a sign error.

**Stage 4 -- Cross-validation**: do aggregates across sources agree? Does today's total volume roughly match historical patterns? This catches systemic issues: a source that is sending stale data, a calculation that silently changed, a duplicate source that is inflating totals.

### Severity levels: warn vs block

Not every quality issue should halt the pipeline. The distinction between warnings and blocking errors is critical:

- **Block**: missing required fields, wrong data types, prices that are clearly impossible (negative, zero). These indicate corrupted data that would produce wrong results.
- **Warn**: slightly unusual volume, a few records with null optional fields, minor timestamp skew. These might be normal variation or early signals of a developing problem.

A pipeline that blocks on every minor anomaly will never run. A pipeline that warns on everything but blocks on nothing will silently ingest garbage. The art is in tuning the thresholds.

In [ ]:
# --- Multi-stage data quality gate ---

@dataclass
class QualityResult:
    stage: str
    passed: bool
    severity: str  # "block" or "warn"
    details: str


class TradeQualityGate:
    """
    Multi-stage quality gate for trade records.
    Each stage produces a list of QualityResults.
    A single "block" result halts the pipeline.
    """

    REQUIRED_FIELDS = {"trade_id", "symbol", "price", "volume", "side", "event_time"}
    EXPECTED_SYMBOLS = {"EURUSD", "USDJPY", "GBPUSD", "AUDUSD"}
    PRICE_BOUNDS = {"EURUSD": (0.5, 2.0), "USDJPY": (80, 200), "GBPUSD": (0.5, 2.5), "AUDUSD": (0.3, 1.5)}
    MAX_VOLUME = 1_000_000

    def run_all(self, records: list[dict]) -> dict:
        """Run all quality stages and return a summary."""
        results: list[QualityResult] = []
        results.extend(self.check_schema(records))
        results.extend(self.check_completeness(records))
        results.extend(self.check_reasonableness(records))

        blocks = [r for r in results if r.severity == "block" and not r.passed]
        warnings = [r for r in results if r.severity == "warn" and not r.passed]
        passed_all = len(blocks) == 0

        return {
            "passed": passed_all,
            "total_checks": len(results),
            "blocks": len(blocks),
            "warnings": len(warnings),
            "details": results,
        }

    def check_schema(self, records: list[dict]) -> list[QualityResult]:
        """Stage 1: required fields and basic types."""
        results = []
        for i, rec in enumerate(records):
            missing = self.REQUIRED_FIELDS - set(rec.keys())
            if missing:
                results.append(QualityResult(
                    stage="schema", passed=False, severity="block",
                    details=f"Record {i}: missing fields {missing}"
                ))
            if "price" in rec:
                try:
                    float(rec["price"])
                except (ValueError, TypeError):
                    results.append(QualityResult(
                        stage="schema", passed=False, severity="block",
                        details=f"Record {i}: price is not numeric: {rec['price']}"
                    ))
        if not results:
            results.append(QualityResult(stage="schema", passed=True, severity="block", details="All records have valid schema"))
        return results

    def check_completeness(self, records: list[dict]) -> list[QualityResult]:
        """Stage 2: expected symbols present, no unexpected nulls."""
        results = []
        observed = {rec.get("symbol") for rec in records}
        missing_symbols = self.EXPECTED_SYMBOLS - observed
        if missing_symbols:
            results.append(QualityResult(
                stage="completeness", passed=False, severity="warn",
                details=f"Missing expected symbols: {missing_symbols}"
            ))

        null_count = sum(1 for rec in records if rec.get("price") is None)
        if null_count > 0:
            results.append(QualityResult(
                stage="completeness", passed=False, severity="block",
                details=f"{null_count} records have null price"
            ))

        if len(records) == 0:
            results.append(QualityResult(
                stage="completeness", passed=False, severity="block",
                details="No records in batch"
            ))

        if not results:
            results.append(QualityResult(stage="completeness", passed=True, severity="warn", details="Completeness OK"))
        return results

    def check_reasonableness(self, records: list[dict]) -> list[QualityResult]:
        """Stage 3: prices in range, volumes positive, timestamps not future."""
        results = []
        now = datetime.now(timezone.utc)

        for i, rec in enumerate(records):
            symbol = rec.get("symbol", "")
            try:
                price = float(rec.get("price", 0))
            except (ValueError, TypeError):
                continue  # schema check already caught this

            bounds = self.PRICE_BOUNDS.get(symbol)
            if bounds and not (bounds[0] <= price <= bounds[1]):
                results.append(QualityResult(
                    stage="reasonableness", passed=False, severity="block",
                    details=f"Record {i}: {symbol} price {price} outside [{bounds[0]}, {bounds[1]}]"
                ))

            volume = rec.get("volume", 0)
            if isinstance(volume, (int, float)) and volume <= 0:
                results.append(QualityResult(
                    stage="reasonableness", passed=False, severity="block",
                    details=f"Record {i}: volume must be positive, got {volume}"
                ))
            elif isinstance(volume, (int, float)) and volume > self.MAX_VOLUME:
                results.append(QualityResult(
                    stage="reasonableness", passed=False, severity="warn",
                    details=f"Record {i}: unusually high volume {volume}"
                ))

            event_time_str = rec.get("event_time", "")
            if event_time_str:
                try:
                    event_time = datetime.fromisoformat(event_time_str)
                    if event_time > now:
                        results.append(QualityResult(
                            stage="reasonableness", passed=False, severity="warn",
                            details=f"Record {i}: event_time {event_time_str} is in the future"
                        ))
                except ValueError:
                    pass  # schema check handles format issues

        if not results:
            results.append(QualityResult(stage="reasonableness", passed=True, severity="block", details="Reasonableness OK"))
        return results


# --- Demonstrate quality gates ---

test_records = [
    {"trade_id": "T001", "symbol": "EURUSD", "price": "1.0845", "volume": 1000, "side": "BUY", "event_time": "2026-04-04T08:15:00+00:00"},
    {"trade_id": "T002", "symbol": "USDJPY", "price": "145.12", "volume": 500, "side": "SELL", "event_time": "2026-04-04T08:15:01+00:00"},
    {"trade_id": "T003", "symbol": "GBPUSD", "price": "0.001", "volume": -10, "side": "BUY", "event_time": "2026-04-04T08:15:02+00:00"},  # bad price + bad volume
    {"trade_id": "T004", "symbol": "EURUSD", "price": "1.0850", "volume": 2_000_000, "side": "SELL", "event_time": "2026-04-04T08:15:03+00:00"},  # unusually high volume
    {"symbol": "AUDUSD", "price": "0.65", "volume": 100, "side": "BUY", "event_time": "2026-04-04T08:15:04+00:00"},  # missing trade_id
]

gate = TradeQualityGate()
report = gate.run_all(test_records)

print(f"=== Quality Gate Report ===")
print(f"  Overall passed: {report['passed']}")
print(f"  Total checks: {report['total_checks']}")
print(f"  Blocking issues: {report['blocks']}")
print(f"  Warnings: {report['warnings']}")
print()
for r in report["details"]:
    status = "PASS" if r.passed else ("BLOCK" if r.severity == "block" else "WARN")
    print(f"  [{status}] {r.stage}: {r.details}")

# Try next:
# 1. Add a cross-validation stage that compares total volume against a historical average.
# 2. Add a "staleness" check: if the newest record is older than 5 minutes, warn.
# 3. Make the price bounds configurable per run instead of hardcoded.

## 5. Dead Letter Queues and Quarantine

When a record fails processing, you have three choices: drop it, retry it, or quarantine it. Dropping is the worst option because you lose visibility into what failed and why. Retrying makes sense for transient errors but not for structurally bad data. Quarantining gives you the best of both worlds: the pipeline keeps running, and the bad records are preserved for investigation and potential reprocessing.

### The DLQ pattern

A dead letter queue (DLQ) is a separate destination (a Kafka topic, a database table, a file) where records that fail processing are sent instead of being dropped or blocking the pipeline. The term comes from postal systems: a letter that cannot be delivered goes to the dead letter office.

### Why not just drop bad records?

- **Lost visibility**: you cannot debug what you cannot see. If you drop bad records silently, you will not know the data is incomplete until someone downstream complains.
- **Cannot diagnose**: without the bad record and the error context, you cannot figure out what went wrong.
- **Cannot recover**: once you fix the underlying issue, you want to replay the quarantined records. If you dropped them, they are gone forever.

### Quarantine design

A good quarantine record stores three things:

1. **The original record**: exactly as received, before any transformation. This lets you replay it.
2. **The error reason**: the specific validation failure, exception message, or processing error. This lets you diagnose.
3. **Metadata**: when it was quarantined, which pipeline stage caught it, what run or batch it belonged to. This lets you query and triage.

### Reprocessing

The quarantine table is not a graveyard. It is a staging area:

1. Query the quarantine table to understand the failure pattern.
2. Fix the underlying issue (schema change, validation bug, upstream data problem).
3. Replay the quarantined records through the pipeline.
4. If they pass this time, remove them from quarantine.
5. If they fail again with a different error, update the quarantine entry.

### Industry context

In Kafka ecosystems, the DLQ is typically a separate topic (e.g., `trades.dlq` alongside `trades`). The consumer reads from the main topic, and if processing fails after retries, it publishes the message to the DLQ topic. A separate monitoring process watches the DLQ topic for volume spikes. In database pipelines, the quarantine is typically a table with the original payload stored as JSON or JSONB alongside error metadata.

In [ ]:
import json
import uuid


# --- Quarantine table implementation ---

@dataclass
class QuarantineEntry:
    quarantine_id: str
    original_record: dict
    error_reason: str
    pipeline_stage: str
    run_id: str
    quarantined_at: str
    reprocessed: bool = False


class QuarantineTable:
    """
    In-memory quarantine table. In production this would be a database table
    or a Kafka DLQ topic.
    """

    def __init__(self):
        self.entries: list[QuarantineEntry] = []

    def quarantine(self, record: dict, error: str, stage: str, run_id: str) -> QuarantineEntry:
        entry = QuarantineEntry(
            quarantine_id=str(uuid.uuid4())[:8],
            original_record=record,
            error_reason=error,
            pipeline_stage=stage,
            run_id=run_id,
            quarantined_at=datetime.now(timezone.utc).isoformat(),
        )
        self.entries.append(entry)
        return entry

    def get_by_stage(self, stage: str) -> list[QuarantineEntry]:
        return [e for e in self.entries if e.pipeline_stage == stage]

    def get_pending(self) -> list[QuarantineEntry]:
        return [e for e in self.entries if not e.reprocessed]

    def mark_reprocessed(self, quarantine_id: str) -> None:
        for e in self.entries:
            if e.quarantine_id == quarantine_id:
                e.reprocessed = True

    def summary(self) -> dict:
        by_stage: dict[str, int] = {}
        for e in self.entries:
            by_stage[e.pipeline_stage] = by_stage.get(e.pipeline_stage, 0) + 1
        return {
            "total": len(self.entries),
            "pending": len(self.get_pending()),
            "reprocessed": len(self.entries) - len(self.get_pending()),
            "by_stage": by_stage,
        }


# --- Processing pipeline with quarantine ---

def process_trade_with_quarantine(
    records: list[dict], quarantine: QuarantineTable, run_id: str
) -> dict:
    """Process records, quarantining failures instead of dropping them."""
    good_records: list[dict] = []
    stats = {"processed": 0, "quarantined": 0}

    for rec in records:
        errors = []

        # Validate required fields
        required = {"trade_id", "symbol", "price", "volume", "side", "event_time"}
        missing = required - set(rec.keys())
        if missing:
            errors.append(f"Missing fields: {missing}")

        # Validate price
        if "price" in rec:
            try:
                price = float(rec["price"])
                if price <= 0:
                    errors.append(f"Non-positive price: {price}")
            except (ValueError, TypeError):
                errors.append(f"Invalid price: {rec['price']}")

        # Validate volume
        if "volume" in rec:
            if not isinstance(rec["volume"], (int, float)) or rec["volume"] <= 0:
                errors.append(f"Invalid volume: {rec['volume']}")

        if errors:
            quarantine.quarantine(
                record=rec,
                error="; ".join(errors),
                stage="validation",
                run_id=run_id,
            )
            stats["quarantined"] += 1
        else:
            good_records.append(rec)
            stats["processed"] += 1

    return {"good_records": good_records, "stats": stats}


# --- Demonstrate quarantine behavior ---

incoming_records = [
    {"trade_id": "T001", "symbol": "EURUSD", "price": "1.0845", "volume": 1000, "side": "BUY", "event_time": "2026-04-04T08:15:00+00:00"},
    {"trade_id": "T002", "symbol": "USDJPY", "price": "bad_price", "volume": 500, "side": "SELL", "event_time": "2026-04-04T08:15:01+00:00"},
    {"trade_id": "T003", "symbol": "GBPUSD", "price": "-1.5", "volume": -10, "side": "BUY", "event_time": "2026-04-04T08:15:02+00:00"},
    {"symbol": "AUDUSD", "price": "0.65", "volume": 100, "side": "BUY", "event_time": "2026-04-04T08:15:03+00:00"},  # missing trade_id
    {"trade_id": "T005", "symbol": "EURUSD", "price": "1.0850", "volume": 800, "side": "SELL", "event_time": "2026-04-04T08:15:04+00:00"},
]

dlq = QuarantineTable()
result = process_trade_with_quarantine(incoming_records, dlq, run_id="run-2026-04-04-001")

print("=== Processing Results ===")
print(f"  Good records: {result['stats']['processed']}")
print(f"  Quarantined: {result['stats']['quarantined']}")

print("\n=== Quarantine Contents ===")
for entry in dlq.entries:
    print(f"  ID: {entry.quarantine_id}")
    print(f"    Record: {json.dumps(entry.original_record, default=str)[:100]}")
    print(f"    Error: {entry.error_reason}")
    print(f"    Stage: {entry.pipeline_stage}")
    print()

print(f"=== Quarantine Summary ===")
print(f"  {dlq.summary()}")

# --- Simulate reprocessing after fix ---
print("\n=== Reprocessing: mark first quarantined record as fixed ===")
if dlq.entries:
    dlq.mark_reprocessed(dlq.entries[0].quarantine_id)
    print(f"  Updated summary: {dlq.summary()}")

# Try next:
# 1. Add a "retry_count" field to QuarantineEntry and increment it on each reprocessing attempt.
# 2. Add a max_retries limit after which the record is marked as "permanently_failed".
# 3. Implement a reprocess_all() method that replays pending entries through the pipeline.

## 6. Lineage and Backfills

Data lineage is the record of where data came from, what transformed it, and where it went. It answers the question every data engineer eventually hears: "This number looks wrong. Where did it come from and what happened to it?"

### Why lineage matters

**Debugging**: when a downstream report shows unexpected numbers, lineage lets you trace backward through every transformation to find where the problem was introduced. Without lineage, debugging becomes archaeology.

**Compliance**: regulations like GDPR and SOX require you to explain how data was processed and where personal data flows. Lineage provides that audit trail.

**Impact analysis**: before changing a table schema, a transformation, or a source, lineage tells you which downstream consumers will be affected. Without it, changes are blind.

### Lineage levels

**Table-level lineage**: tracks which tables or topics feed into which other tables or topics. This is the coarsest level and the easiest to maintain. Example: "The `trade_aggregates` table is derived from the `raw_trades` table."

**Column-level lineage**: tracks which specific columns in the output derive from which columns in the input, and through what transformations. Example: "The `vwap` column in `trade_aggregates` is computed as `sum(price * volume) / sum(volume)` from the `price` and `volume` columns in `raw_trades`." This is more expensive to maintain but much more powerful for debugging.

**Row-level lineage**: tracks which specific input rows contributed to each output row. This is the most expensive but is sometimes necessary for compliance or debugging specific anomalies.

### Backfills

A backfill is the reprocessing of historical data, usually triggered by a bug fix, a schema change, a new derived table, or a data quality issue that affected a range of historical records.

**Backfill safety rules**:

1. **Must be idempotent**: if the backfill crashes and restarts, it should not corrupt data. This is where idempotency from Section 2 pays off directly.
2. **Must not corrupt live data**: the backfill should write to a staging area or use atomic swaps, not update the live table in place while active queries are running.
3. **Must handle time boundaries correctly**: the backfill should process exactly the affected time range, not accidentally reprocess or skip adjacent data.
4. **Must be observable**: you should know how far the backfill has progressed, how many records it has processed, and whether it encountered errors.

### Industry context

Scenario: a researcher at Rakuten reports that volume data looks wrong since Tuesday. With lineage, you can immediately see: raw_trades -> validated_trades -> trade_aggregates. You check the `validated_trades` step and discover that a validation rule was changed on Tuesday that started dropping records with volume above a threshold that was set too low. You fix the rule and backfill the affected date range. Without lineage, you would spend hours guessing which of dozens of transformations introduced the problem.

In [ ]:
# --- Simple lineage tracking with run metadata ---

@dataclass
class LineageRecord:
    run_id: str
    step_name: str
    input_source: str
    output_destination: str
    record_count_in: int
    record_count_out: int
    started_at: str
    completed_at: str
    status: str  # "success", "failed", "partial"
    parameters: dict = field(default_factory=dict)
    error: str | None = None


class LineageTracker:
    """
    Tracks lineage metadata for pipeline runs.
    In production, this would write to a lineage database or service
    like Apache Atlas, OpenLineage, or Marquez.
    """

    def __init__(self):
        self.records: list[LineageRecord] = []

    def record_step(self, **kwargs) -> LineageRecord:
        entry = LineageRecord(**kwargs)
        self.records.append(entry)
        return entry

    def get_upstream(self, destination: str) -> list[LineageRecord]:
        """What feeds into this destination?"""
        return [r for r in self.records if r.output_destination == destination]

    def get_downstream(self, source: str) -> list[LineageRecord]:
        """What does this source feed into?"""
        return [r for r in self.records if r.input_source == source]

    def get_run_history(self, step_name: str) -> list[LineageRecord]:
        """All runs of a specific step."""
        return [r for r in self.records if r.step_name == step_name]

    def print_lineage_graph(self) -> None:
        """Print a simple text-based lineage graph."""
        edges: set[tuple[str, str, str]] = set()
        for r in self.records:
            edges.add((r.input_source, r.step_name, r.output_destination))
        for src, step, dst in sorted(edges):
            print(f"  {src} --[{step}]--> {dst}")


# --- Simulate a multi-step pipeline with lineage ---

tracker = LineageTracker()
run_id = "run-2026-04-04-001"

# Step 1: ingest raw trades
tracker.record_step(
    run_id=run_id,
    step_name="ingest_raw",
    input_source="finnhub_websocket",
    output_destination="raw_trades",
    record_count_in=1000,
    record_count_out=1000,
    started_at="2026-04-04T08:00:00Z",
    completed_at="2026-04-04T08:00:05Z",
    status="success",
    parameters={"symbols": ["EURUSD", "USDJPY", "GBPUSD"]},
)

# Step 2: validate and clean
tracker.record_step(
    run_id=run_id,
    step_name="validate_and_clean",
    input_source="raw_trades",
    output_destination="validated_trades",
    record_count_in=1000,
    record_count_out=985,
    started_at="2026-04-04T08:00:06Z",
    completed_at="2026-04-04T08:00:08Z",
    status="success",
    parameters={"quarantined": 15, "quality_gate": "v2.1"},
)

# Step 3: aggregate VWAP
tracker.record_step(
    run_id=run_id,
    step_name="aggregate_vwap",
    input_source="validated_trades",
    output_destination="trade_aggregates",
    record_count_in=985,
    record_count_out=12,
    started_at="2026-04-04T08:00:09Z",
    completed_at="2026-04-04T08:00:10Z",
    status="success",
    parameters={"window_size": "1min", "symbols": 3},
)

print("=== Lineage Graph ===")
tracker.print_lineage_graph()

print("\n=== What feeds into trade_aggregates? ===")
for r in tracker.get_upstream("trade_aggregates"):
    print(f"  {r.input_source} via {r.step_name} ({r.record_count_in} in -> {r.record_count_out} out)")

print("\n=== What does raw_trades feed into? ===")
for r in tracker.get_downstream("raw_trades"):
    print(f"  {r.output_destination} via {r.step_name} ({r.record_count_in} in -> {r.record_count_out} out)")

print("\n=== Full run history ===")
for r in tracker.records:
    drop_rate = round((1 - r.record_count_out / max(r.record_count_in, 1)) * 100, 1)
    print(f"  {r.step_name}: {r.record_count_in} -> {r.record_count_out} ({drop_rate}% drop, {r.status})")


# --- Backfill simulation ---

print("\n\n=== Backfill Simulation ===")
print("Scenario: validation rule was too aggressive from April 1-3. Need to reprocess.")

backfill_dates = ["2026-04-01", "2026-04-02", "2026-04-03"]
for date in backfill_dates:
    backfill_run_id = f"backfill-{date}"
    tracker.record_step(
        run_id=backfill_run_id,
        step_name="validate_and_clean",
        input_source="raw_trades",
        output_destination="validated_trades",
        record_count_in=950,
        record_count_out=945,  # much less drop with fixed rule
        started_at=f"{date}T00:00:00Z",
        completed_at=f"{date}T00:00:03Z",
        status="success",
        parameters={"quality_gate": "v2.2_fixed", "backfill": True, "date_range": date},
    )
    print(f"  Backfilled {date}: 950 in -> 945 out (v2.2_fixed)")

print(f"\n  Total lineage records: {len(tracker.records)}")

# Try next:
# 1. Add column-level lineage: track which fields the VWAP step reads from validated_trades.
# 2. Implement a "lineage diff" that compares two runs of the same step and highlights changes.
# 3. Add a check that prevents backfill from processing dates outside the specified range.

## 7. CDC (Change Data Capture)

Change Data Capture is the practice of identifying and capturing row-level changes (inserts, updates, deletes) from a source database so that downstream systems can react to them. Instead of periodically dumping the entire table, CDC streams only the changes, which is both faster and more efficient.

### Why CDC matters

Without CDC, the standard approach to synchronizing data between systems is a full table scan on a schedule: every hour, read the entire source table and compare it to the destination. This works but has serious limitations:

- **Latency**: data is stale until the next full scan completes.
- **Resource cost**: scanning millions of rows every hour wastes database I/O.
- **Missed deletes**: a full scan can detect new and changed rows, but a row that was deleted since the last scan is invisible unless you do a full comparison.

CDC solves all three: changes are captured as they happen, only changed rows are transmitted, and deletes are explicitly captured.

### CDC approaches

**Log-based CDC**: read the database's write-ahead log (WAL in PostgreSQL, binlog in MySQL). Every committed change is already recorded in the WAL for crash recovery. CDC tools like Debezium tap into this log and stream changes to Kafka. This is the gold standard because it captures all changes including deletes, has minimal impact on the source database, and provides ordering guarantees.

**Trigger-based CDC**: install database triggers that fire on INSERT, UPDATE, and DELETE, writing change records to a shadow table. The shadow table is then read by the CDC consumer. This captures everything but adds overhead to every write operation on the source database and requires schema changes.

**Timestamp-based CDC**: query `WHERE updated_at > last_run_timestamp`. This is the simplest approach and requires no special database features, but it has a critical limitation: it cannot capture deletes (a deleted row has no `updated_at`), and it can miss updates if the clock skews or if `updated_at` is not reliably maintained.

### Choosing an approach

| Approach | Captures deletes? | Source impact | Complexity | Latency |
|----------|-------------------|---------------|------------|---------|
| Log-based | Yes | Minimal | Medium (Debezium setup) | Seconds |
| Trigger-based | Yes | High (trigger overhead) | Medium | Seconds |
| Timestamp-based | No | Minimal | Low | Minutes to hours |

### Industry context

In the energy trading platform, if reference data (instrument metadata, venue configs, trading schedules) lives in a PostgreSQL database, CDC via Debezium can stream changes to Kafka topics. Downstream consumers subscribe to these topics and update their local caches in near-real-time, rather than running expensive periodic full refreshes.

In [ ]:
# --- Simulate timestamp-based CDC ---

class SourceDatabase:
    """Simulates a source database with an updated_at column."""

    def __init__(self):
        self.table: dict[str, dict] = {}  # keyed by primary key

    def upsert(self, pk: str, data: dict, timestamp: str) -> None:
        self.table[pk] = {**data, "pk": pk, "updated_at": timestamp}

    def delete(self, pk: str) -> dict | None:
        return self.table.pop(pk, None)

    def query_changes_since(self, since: str) -> list[dict]:
        """Timestamp-based CDC: find rows updated after 'since'."""
        return [
            row for row in self.table.values()
            if row["updated_at"] > since
        ]

    def full_snapshot(self) -> list[dict]:
        return list(self.table.values())


class CDCConsumer:
    """Consumes CDC events and maintains a local replica."""

    def __init__(self):
        self.replica: dict[str, dict] = {}
        self.last_sync: str = "1970-01-01T00:00:00Z"
        self.change_log: list[dict] = []

    def apply_changes(self, changes: list[dict]) -> dict:
        stats = {"inserts": 0, "updates": 0}
        for change in changes:
            pk = change["pk"]
            if pk in self.replica:
                stats["updates"] += 1
                action = "UPDATE"
            else:
                stats["inserts"] += 1
                action = "INSERT"
            self.replica[pk] = change
            self.change_log.append({"action": action, "pk": pk, "ts": change["updated_at"]})
            if change["updated_at"] > self.last_sync:
                self.last_sync = change["updated_at"]
        return stats


# --- Simulate CDC workflow ---

source = SourceDatabase()
consumer = CDCConsumer()

# Initial load: several instruments
print("=== Step 1: Initial data ===")
source.upsert("EURUSD", {"symbol": "EURUSD", "name": "Euro/USD", "lot_size": 100000}, "2026-04-04T08:00:00Z")
source.upsert("USDJPY", {"symbol": "USDJPY", "name": "USD/Yen", "lot_size": 100000}, "2026-04-04T08:00:00Z")
source.upsert("GBPUSD", {"symbol": "GBPUSD", "name": "GBP/USD", "lot_size": 100000}, "2026-04-04T08:00:00Z")

changes = source.query_changes_since(consumer.last_sync)
stats = consumer.apply_changes(changes)
print(f"  Changes found: {len(changes)}, Applied: {stats}")
print(f"  Replica size: {len(consumer.replica)}")

# Step 2: Some updates and a new instrument
print("\n=== Step 2: Updates + new instrument ===")
source.upsert("EURUSD", {"symbol": "EURUSD", "name": "Euro/USD", "lot_size": 200000}, "2026-04-04T09:00:00Z")  # changed lot_size
source.upsert("AUDUSD", {"symbol": "AUDUSD", "name": "AUD/USD", "lot_size": 100000}, "2026-04-04T09:00:01Z")  # new

changes = source.query_changes_since(consumer.last_sync)
stats = consumer.apply_changes(changes)
print(f"  Changes found: {len(changes)}, Applied: {stats}")
print(f"  EURUSD lot_size in replica: {consumer.replica['EURUSD']['lot_size']}")

# Step 3: Delete -- demonstrate the timestamp-based CDC limitation
print("\n=== Step 3: Delete (timestamp CDC limitation) ===")
deleted = source.delete("GBPUSD")
print(f"  Deleted GBPUSD from source: {deleted is not None}")

changes = source.query_changes_since(consumer.last_sync)
print(f"  Changes found by timestamp CDC: {len(changes)} (delete is invisible!)")
print(f"  GBPUSD still in replica: {'GBPUSD' in consumer.replica}")
print(f"  Source has GBPUSD: {'GBPUSD' in source.table}")
print("  --> This is why timestamp-based CDC cannot capture deletes.")

# Show how log-based CDC would handle this
print("\n=== How log-based CDC would handle the delete ===")
print("  Debezium would emit: {'op': 'd', 'before': {'pk': 'GBPUSD', ...}, 'after': null}")
print("  The consumer would delete GBPUSD from the replica.")
print("  Log-based CDC captures ALL operations: INSERT, UPDATE, DELETE.")

print("\n=== Change log ===")
for entry in consumer.change_log:
    print(f"  {entry}")

# Try next:
# 1. Implement a full-comparison CDC that detects deletes by comparing source snapshot to replica.
# 2. Add a "soft delete" column (deleted_at) and modify the timestamp CDC to detect it.
# 3. Simulate a log-based CDC by maintaining an explicit change log in the source database.

## 8. Batch vs Stream Tradeoffs

This is one of the most important architectural decisions in data engineering, and it is often made badly because people either default to what they know or chase the latest technology without understanding the tradeoffs.

### Batch processing

Batch processing means collecting data over a period, then processing it all at once on a schedule. Airflow triggering a Spark job every hour is the canonical example.

**Strengths**:
- **Simple to reason about**: the input is a finite, bounded dataset. You can inspect it, reprocess it, debug it.
- **Simple to debug**: if the batch fails, you know exactly which batch, what data, and what time range.
- **Efficient for large historical reprocessing**: when you need to backfill months of data, batch is natural.
- **Mature tooling**: Airflow, dbt, Spark batch mode have years of production hardening.

**Weaknesses**:
- **Latency**: data is stale until the next batch completes. A 1-hour batch means data can be up to 1 hour old.
- **Resource spikes**: a large batch job can spike CPU and memory, then sit idle until the next run.
- **Hard to scale down**: even if you only have 10 new records, the batch infrastructure still runs.

### Stream processing

Stream processing means processing data continuously as it arrives. Kafka Streams, Flink, and Spark Structured Streaming are the main tools.

**Strengths**:
- **Low latency**: events are processed within seconds of arrival.
- **Natural for event-driven systems**: if your business logic is "when X happens, do Y", streaming fits perfectly.
- **Even resource usage**: processing is spread over time instead of concentrated in spikes.

**Weaknesses**:
- **Harder to debug**: the input is an unbounded stream. You cannot easily "replay this specific batch."
- **Harder to reprocess**: backfilling requires replaying from a log (Kafka offset reset), which has its own complexity.
- **Ordering guarantees are complex**: in-order processing within a partition is straightforward, but cross-partition ordering requires careful design.
- **State management**: windowed aggregations, joins, and exactly-once require stateful processing, which adds significant operational complexity.

### Lambda and Kappa architectures

**Lambda architecture**: run both a batch layer and a stream layer in parallel. The stream layer handles real-time with approximate results. The batch layer periodically recomputes exact results. A serving layer merges both. This gives you low latency AND eventual accuracy, but at the cost of maintaining two codepaths that must produce compatible results.

**Kappa architecture**: stream-only, with a replayable log (Kafka) as the source of truth. When you need to reprocess, you reset the consumer offset and replay from the log. This is simpler than Lambda but requires your streaming infrastructure to handle both real-time and historical replay loads.

### Decision framework

Start with batch unless at least one of these is true:
- Business requires sub-minute latency (trading signals, fraud detection, real-time dashboards)
- The data naturally arrives as events (clickstream, IoT sensors, market data feeds)
- The workload is continuous and does not have natural batch boundaries

### Industry context

Your energy trading platform uses Kafka streaming for VWAP calculation. This is the right choice because:
- Trade data arrives continuously from WebSocket feeds
- VWAP needs to be current (stale VWAP is useless for trading decisions)
- The data is naturally event-driven (each trade is an event)
- 1-minute tumbling windows map cleanly to stream processing primitives

But for end-of-day reporting, historical analytics, and compliance exports, batch processing (e.g., an Airflow job reading from TimescaleDB) would be more appropriate because those workloads do not need sub-minute latency and benefit from the simplicity of batch.

In [ ]:
# --- Compare batch vs stream processing for the same workload ---

from collections import defaultdict
from decimal import Decimal, ROUND_HALF_UP


def generate_mock_trades(n: int = 20) -> list[dict]:
    """Generate mock trade events with realistic timestamps."""
    random.seed(42)
    symbols = ["EURUSD", "USDJPY", "GBPUSD"]
    base_prices = {"EURUSD": 1.0845, "USDJPY": 145.12, "GBPUSD": 1.2641}
    trades = []
    for i in range(n):
        symbol = random.choice(symbols)
        price = round(base_prices[symbol] * random.uniform(0.999, 1.001), 5)
        volume = random.randint(100, 5000)
        minute = i // 5  # roughly 5 trades per minute
        ts = f"2026-04-04T08:{minute:02d}:{(i % 5) * 12:02d}+00:00"
        trades.append({
            "trade_id": f"T{i:04d}",
            "symbol": symbol,
            "price": price,
            "volume": volume,
            "event_time": ts,
        })
    return trades


# --- Batch approach: process all at once, group by window ---

def batch_vwap(trades: list[dict]) -> dict:
    """
    Batch VWAP: collect all trades, group by (symbol, minute window),
    compute VWAP for each group.
    """
    buckets: dict[tuple, list] = defaultdict(list)
    for t in trades:
        minute = t["event_time"][:16]  # truncate to minute
        key = (t["symbol"], minute)
        buckets[key].append(t)

    results = {}
    for (symbol, window), window_trades in sorted(buckets.items()):
        total_pv = sum(t["price"] * t["volume"] for t in window_trades)
        total_vol = sum(t["volume"] for t in window_trades)
        vwap = round(total_pv / total_vol, 6) if total_vol > 0 else 0
        results[(symbol, window)] = {
            "vwap": vwap,
            "total_volume": total_vol,
            "trade_count": len(window_trades),
        }
    return results


# --- Stream approach: process one at a time, maintain running state ---

class StreamVWAP:
    """
    Stream VWAP: process trades one at a time, maintain running
    price*volume sum and volume sum per (symbol, minute).
    Emit updated VWAP after each trade.
    """

    def __init__(self):
        self.state: dict[tuple, dict] = defaultdict(
            lambda: {"sum_pv": 0.0, "sum_vol": 0, "count": 0}
        )
        self.emissions: list[dict] = []

    def process(self, trade: dict) -> dict:
        minute = trade["event_time"][:16]
        key = (trade["symbol"], minute)
        s = self.state[key]
        s["sum_pv"] += trade["price"] * trade["volume"]
        s["sum_vol"] += trade["volume"]
        s["count"] += 1
        vwap = round(s["sum_pv"] / s["sum_vol"], 6) if s["sum_vol"] > 0 else 0
        result = {
            "symbol": trade["symbol"],
            "window": minute,
            "vwap": vwap,
            "total_volume": s["sum_vol"],
            "trade_count": s["count"],
        }
        self.emissions.append(result)
        return result


# --- Run both and compare ---

mock_trades = generate_mock_trades(20)

# Batch
print("=== Batch VWAP (process all at once) ===")
batch_start = time.perf_counter()
batch_results = batch_vwap(mock_trades)
batch_elapsed = time.perf_counter() - batch_start
for key, val in list(batch_results.items())[:6]:
    print(f"  {key[0]} @ {key[1]}: VWAP={val['vwap']}, vol={val['total_volume']}, trades={val['trade_count']}")
print(f"  Computed in {batch_elapsed*1000:.3f}ms (all at once)")

# Stream
print("\n=== Stream VWAP (process one at a time) ===")
stream = StreamVWAP()
stream_start = time.perf_counter()
for trade in mock_trades:
    result = stream.process(trade)
stream_elapsed = time.perf_counter() - stream_start
# Show the final state per window (equivalent to batch output)
final_state = {}
for emission in stream.emissions:
    key = (emission["symbol"], emission["window"])
    final_state[key] = emission
for key, val in list(sorted(final_state.items()))[:6]:
    print(f"  {val['symbol']} @ {val['window']}: VWAP={val['vwap']}, vol={val['total_volume']}, trades={val['trade_count']}")
print(f"  Computed in {stream_elapsed*1000:.3f}ms (incremental)")

# Key difference
print("\n=== Key difference ===")
print(f"  Batch: waited for all {len(mock_trades)} trades, then computed everything.")
print(f"  Stream: emitted {len(stream.emissions)} intermediate results (one per trade).")
print(f"  Stream gave the first result after 1 trade. Batch gave results only after all 20.")
print(f"  Both produce the same final VWAP values.")

# Verify equivalence
mismatches = 0
for key in batch_results:
    if key in final_state:
        if batch_results[key]["vwap"] != final_state[key]["vwap"]:
            mismatches += 1
print(f"  VWAP mismatches between batch and stream: {mismatches}")

# Try next:
# 1. Add a "late arrival" trade (timestamp from a previous window) and see how each approach handles it.
# 2. Implement a grace period in the stream processor that keeps windows open for late data.
# 3. Compare memory usage: batch must hold all trades; stream only holds running sums.

## 9. Observability: SLIs, SLOs, and Operational Metrics

Observability is the ability to understand the internal state of a system by examining its external outputs. For data pipelines, this means knowing whether your data is fresh, complete, correct, and delivered on time -- without having to SSH into a server and grep through logs.

### SLI, SLO, SLA: the hierarchy

**SLI (Service Level Indicator)**: a measurable signal of system health. It is a number you can compute. Examples: ingestion latency in seconds, error rate as a percentage, data freshness in minutes.

**SLO (Service Level Objective)**: a target for an SLI. It is a threshold that defines "good enough." Examples: "99.9% of ingestion runs complete within 5 minutes", "data freshness is less than 2 minutes during trading hours", "error rate stays below 0.1%."

**SLA (Service Level Agreement)**: a contractual commitment, usually external-facing, with consequences for breach (refunds, penalties, escalation). SLAs are business agreements built on top of SLOs.

The relationship is: SLIs are what you measure, SLOs are what you target, SLAs are what you promise. Engineers own SLIs and SLOs. Business owners negotiate SLAs.

### Key metrics for data pipelines

**Freshness**: how old is the newest record in the destination? If the newest trade in your database is 10 minutes old but trades are flowing continuously, something is wrong. Freshness is often the first thing to degrade when a pipeline breaks.

**Completeness**: are all expected sources and symbols present in the latest batch? If you expect data for 4 symbols and only see 3, a source might be down. Completeness catches silent data loss.

**Volume**: is the record count within the expected range? A sudden drop to zero is obvious. A 30% drop might be a real market condition (quiet trading day) or a bug. Historical baselines help distinguish.

**Latency**: how long from event time (when the trade happened) to available time (when it appears in the destination)? This measures end-to-end pipeline speed. High latency can mean backpressure, slow consumers, or resource contention.

**Error rate**: what fraction of records fail validation or processing? A steady 0.01% error rate might be normal (bad source data). A sudden spike to 5% suggests a systemic issue.

### Alerting: alert on SLO breach, not on every transient error

The biggest mistake in pipeline alerting is alerting on every error. A single transient failure that auto-recovers should not page anyone. Instead:

- Define SLOs with error budgets: "99.9% success rate means we tolerate 0.1% errors."
- Alert when the SLO is in danger of being breached, not when a single request fails.
- Use burn-rate alerts: "at the current error rate, we will exhaust our monthly error budget in 2 hours."
- Separate fast-burn alerts (something broke right now) from slow-burn alerts (gradual degradation).

### Industry context

For the energy trading platform, meaningful SLOs might be:
- Freshness: "During trading hours (08:00-16:00 UTC), the newest trade in TimescaleDB is less than 60 seconds old, 99.5% of the time."
- Completeness: "All configured symbols have at least one trade per 5-minute window during trading hours."
- Latency: "End-to-end latency from Kafka message to TimescaleDB row is under 5 seconds, 99th percentile."
- Error rate: "Less than 0.1% of trade messages are quarantined per hour."

In [ ]:
# --- SLI/SLO definitions and evaluation ---

@dataclass
class SLIDefinition:
    name: str
    description: str
    unit: str
    compute: str  # human-readable computation


@dataclass
class SLODefinition:
    sli_name: str
    target: str
    threshold: float
    comparison: str  # "lt" (less than), "gt" (greater than), "between"
    window: str  # evaluation window


@dataclass
class SLOEvaluation:
    slo: SLODefinition
    current_value: float
    passed: bool
    margin: float  # how far from threshold (positive = good)


class PipelineObservability:
    """
    Define and evaluate SLIs/SLOs for a trade data ingestion pipeline.
    """

    def __init__(self):
        self.slis = [
            SLIDefinition("freshness", "Age of newest record in seconds", "seconds",
                          "now() - max(event_time) from destination"),
            SLIDefinition("completeness", "Fraction of expected symbols present", "ratio",
                          "count(distinct observed symbols) / count(expected symbols)"),
            SLIDefinition("volume", "Records ingested per window", "count",
                          "count(*) from destination WHERE event_time in window"),
            SLIDefinition("latency_p99", "99th percentile end-to-end latency", "seconds",
                          "percentile(0.99, ingested_at - event_time)"),
            SLIDefinition("error_rate", "Fraction of records quarantined", "ratio",
                          "quarantined / (processed + quarantined)"),
        ]
        self.slos = [
            SLODefinition("freshness", "< 60s during trading hours", 60.0, "lt", "5min"),
            SLODefinition("completeness", "> 0.95 symbol coverage", 0.95, "gt", "5min"),
            SLODefinition("volume", "> 10 records per minute", 10.0, "gt", "1min"),
            SLODefinition("latency_p99", "< 5s end-to-end", 5.0, "lt", "5min"),
            SLODefinition("error_rate", "< 0.001 quarantine rate", 0.001, "lt", "1hour"),
        ]

    def evaluate(self, metrics: dict[str, float]) -> list[SLOEvaluation]:
        results = []
        for slo in self.slos:
            value = metrics.get(slo.sli_name, 0)
            if slo.comparison == "lt":
                passed = value < slo.threshold
                margin = slo.threshold - value
            elif slo.comparison == "gt":
                passed = value > slo.threshold
                margin = value - slo.threshold
            else:
                passed = False
                margin = 0
            results.append(SLOEvaluation(slo=slo, current_value=value, passed=passed, margin=round(margin, 4)))
        return results


# --- Simulate pipeline metrics and evaluate SLOs ---

observability = PipelineObservability()

print("=== SLI Definitions ===")
for sli in observability.slis:
    print(f"  {sli.name} ({sli.unit}): {sli.description}")
    print(f"    Computed as: {sli.compute}")

# Scenario 1: healthy pipeline
print("\n=== Scenario 1: Healthy Pipeline ===")
healthy_metrics = {
    "freshness": 12.5,     # 12.5 seconds old
    "completeness": 1.0,   # all symbols present
    "volume": 145,         # 145 records/minute
    "latency_p99": 2.3,    # 2.3s p99 latency
    "error_rate": 0.0002,  # 0.02% error rate
}

for ev in observability.evaluate(healthy_metrics):
    status = "PASS" if ev.passed else "FAIL"
    print(f"  [{status}] {ev.slo.sli_name}: {ev.current_value} (target: {ev.slo.target}, margin: {ev.margin})")

# Scenario 2: degraded pipeline
print("\n=== Scenario 2: Degraded Pipeline ===")
degraded_metrics = {
    "freshness": 95.0,     # 95 seconds -- stale!
    "completeness": 0.75,  # only 3 of 4 symbols
    "volume": 45,          # low but above threshold
    "latency_p99": 8.7,    # high latency
    "error_rate": 0.05,    # 5% error rate -- bad!
}

for ev in observability.evaluate(degraded_metrics):
    status = "PASS" if ev.passed else "FAIL"
    print(f"  [{status}] {ev.slo.sli_name}: {ev.current_value} (target: {ev.slo.target}, margin: {ev.margin})")

# Count breaches
breaches = [ev for ev in observability.evaluate(degraded_metrics) if not ev.passed]
print(f"\n  SLO breaches: {len(breaches)}/{len(observability.slos)}")
for b in breaches:
    print(f"    {b.slo.sli_name}: current={b.current_value}, threshold={b.slo.threshold}")

# --- Error budget calculation ---
print("\n=== Error Budget Example ===")
slo_target = 0.999  # 99.9% success rate
monthly_minutes = 30 * 24 * 60  # ~43,200 minutes
error_budget_minutes = monthly_minutes * (1 - slo_target)
print(f"  SLO: {slo_target*100}% success rate")
print(f"  Monthly error budget: {error_budget_minutes:.1f} minutes of allowed downtime")
print(f"  That is {error_budget_minutes/60:.1f} hours per month")
current_downtime_minutes = 15
remaining = error_budget_minutes - current_downtime_minutes
print(f"  Current month downtime: {current_downtime_minutes} minutes")
print(f"  Remaining budget: {remaining:.1f} minutes ({remaining/error_budget_minutes*100:.1f}%)")

# Try next:
# 1. Add a "burn rate" calculation: if we keep failing at the current rate, when do we exhaust the budget?
# 2. Define SLOs for a different pipeline (e.g., end-of-day reporting) with different thresholds.
# 3. Implement a simple alerting rule that triggers when margin goes negative.

## 10. Mini Lab: Complete Reliable Ingestion Pipeline

This lab ties together everything from the previous sections into a single working pipeline. The pipeline:

1. Reads mock source data (simulating a batch from a market data provider)
2. Validates with quality gates (schema, completeness, reasonableness)
3. Quarantines bad records (preserving them for investigation)
4. Deduplicates with idempotency keys (safe for reruns)
5. Tracks lineage metadata (who, what, when, how many)
6. Emits run metrics (freshness, completeness, error rate, volume)

This is the "reliable ingestion prototype" from the build milestones. Every piece has been explained and demonstrated individually above. Now we combine them.

In [ ]:
# --- Complete reliable ingestion pipeline ---

class ReliableIngestionPipeline:
    """
    End-to-end pipeline combining:
    - Quality gates (schema + completeness + reasonableness)
    - Quarantine (bad records preserved with error context)
    - Idempotency (dedup via composite key)
    - Lineage (run metadata tracking)
    - Metrics (SLI computation)
    """

    REQUIRED_FIELDS = {"trade_id", "symbol", "price", "volume", "side", "event_time", "source"}
    EXPECTED_SYMBOLS = {"EURUSD", "USDJPY", "GBPUSD", "AUDUSD"}
    PRICE_BOUNDS = {"EURUSD": (0.5, 2.0), "USDJPY": (80, 200), "GBPUSD": (0.5, 2.5), "AUDUSD": (0.3, 1.5)}

    def __init__(self):
        self.store: list[dict] = []
        self.seen_keys: set[str] = set()
        self.quarantine: list[dict] = []
        self.lineage: list[dict] = []
        self.run_count = 0

    def run(self, records: list[dict], run_label: str = "") -> dict:
        """Execute the full pipeline and return a comprehensive report."""
        self.run_count += 1
        run_id = run_label or f"run-{self.run_count:03d}"
        start_time = datetime.now(timezone.utc)

        stats = {
            "run_id": run_id,
            "input_count": len(records),
            "schema_failures": 0,
            "reasonableness_failures": 0,
            "duplicates_skipped": 0,
            "ingested": 0,
            "quarantined": 0,
        }

        validated: list[dict] = []

        # --- Stage 1: Schema validation ---
        for rec in records:
            missing = self.REQUIRED_FIELDS - set(rec.keys())
            if missing:
                self._quarantine(rec, f"Missing fields: {missing}", "schema", run_id)
                stats["schema_failures"] += 1
                stats["quarantined"] += 1
                continue

            try:
                price = float(rec["price"])
            except (ValueError, TypeError):
                self._quarantine(rec, f"Invalid price: {rec['price']}", "schema", run_id)
                stats["schema_failures"] += 1
                stats["quarantined"] += 1
                continue

            if not isinstance(rec["volume"], (int, float)) or rec["volume"] <= 0:
                self._quarantine(rec, f"Invalid volume: {rec['volume']}", "schema", run_id)
                stats["schema_failures"] += 1
                stats["quarantined"] += 1
                continue

            validated.append(rec)

        # --- Stage 2: Reasonableness checks ---
        reasonable: list[dict] = []
        for rec in validated:
            price = float(rec["price"])
            symbol = rec["symbol"]
            bounds = self.PRICE_BOUNDS.get(symbol)

            if bounds and not (bounds[0] <= price <= bounds[1]):
                self._quarantine(rec, f"Price {price} outside bounds {bounds}", "reasonableness", run_id)
                stats["reasonableness_failures"] += 1
                stats["quarantined"] += 1
                continue

            reasonable.append(rec)

        # --- Stage 3: Deduplication ---
        for rec in reasonable:
            key = f"{rec['source']}|{rec['trade_id']}|{rec['event_time']}"
            if key in self.seen_keys:
                stats["duplicates_skipped"] += 1
                continue
            self.seen_keys.add(key)
            self.store.append(rec)
            stats["ingested"] += 1

        # --- Lineage ---
        end_time = datetime.now(timezone.utc)
        self.lineage.append({
            "run_id": run_id,
            "input_source": "mock_market_data",
            "output_destination": "trade_store",
            "records_in": stats["input_count"],
            "records_out": stats["ingested"],
            "quarantined": stats["quarantined"],
            "duplicates": stats["duplicates_skipped"],
            "started_at": start_time.isoformat(),
            "completed_at": end_time.isoformat(),
            "duration_ms": round((end_time - start_time).total_seconds() * 1000, 2),
        })

        # --- Metrics ---
        observed_symbols = {r["symbol"] for r in self.store}
        completeness = len(observed_symbols & self.EXPECTED_SYMBOLS) / len(self.EXPECTED_SYMBOLS)
        total_processed = stats["ingested"] + stats["quarantined"] + stats["duplicates_skipped"]
        error_rate = stats["quarantined"] / max(total_processed, 1)

        stats["metrics"] = {
            "completeness": round(completeness, 4),
            "error_rate": round(error_rate, 4),
            "volume": stats["ingested"],
            "store_total": len(self.store),
            "quarantine_total": len(self.quarantine),
        }

        return stats

    def _quarantine(self, record: dict, error: str, stage: str, run_id: str) -> None:
        self.quarantine.append({
            "record": record,
            "error": error,
            "stage": stage,
            "run_id": run_id,
            "quarantined_at": datetime.now(timezone.utc).isoformat(),
        })


# --- Mock source data ---

source_batch_1 = [
    {"trade_id": "T001", "symbol": "EURUSD", "price": "1.0845", "volume": 1000, "side": "BUY", "event_time": "2026-04-04T08:15:00Z", "source": "finnhub"},
    {"trade_id": "T002", "symbol": "USDJPY", "price": "145.12", "volume": 500, "side": "SELL", "event_time": "2026-04-04T08:15:01Z", "source": "finnhub"},
    {"trade_id": "T003", "symbol": "GBPUSD", "price": "1.2641", "volume": 750, "side": "BUY", "event_time": "2026-04-04T08:15:02Z", "source": "finnhub"},
    {"trade_id": "T004", "symbol": "AUDUSD", "price": "0.6510", "volume": 300, "side": "SELL", "event_time": "2026-04-04T08:15:03Z", "source": "finnhub"},
    {"trade_id": "T005", "symbol": "EURUSD", "price": "bad", "volume": 100, "side": "BUY", "event_time": "2026-04-04T08:15:04Z", "source": "finnhub"},  # bad price
    {"trade_id": "T006", "symbol": "USDJPY", "price": "5.0", "volume": 200, "side": "BUY", "event_time": "2026-04-04T08:15:05Z", "source": "finnhub"},  # price out of range
    {"symbol": "EURUSD", "price": "1.0846", "volume": 400, "side": "SELL", "event_time": "2026-04-04T08:15:06Z", "source": "finnhub"},  # missing trade_id
]

# --- Run 1: initial ingestion ---

pipeline = ReliableIngestionPipeline()
print("=" * 60)
print("RUN 1: Initial Ingestion")
print("=" * 60)
report = pipeline.run(source_batch_1, "initial-2026-04-04")
print(f"  Input: {report['input_count']}")
print(f"  Ingested: {report['ingested']}")
print(f"  Quarantined: {report['quarantined']} (schema: {report['schema_failures']}, bounds: {report['reasonableness_failures']})")
print(f"  Duplicates skipped: {report['duplicates_skipped']}")
print(f"  Metrics: {report['metrics']}")

# --- Run 2: replay same batch (idempotency test) ---

print(f"\n{'=' * 60}")
print("RUN 2: Replay Same Batch (idempotency test)")
print("=" * 60)
report2 = pipeline.run(source_batch_1, "replay-2026-04-04")
print(f"  Input: {report2['input_count']}")
print(f"  Ingested: {report2['ingested']} (should be 0 -- all deduped)")
print(f"  Duplicates skipped: {report2['duplicates_skipped']}")
print(f"  Store total unchanged: {report2['metrics']['store_total']}")

# --- Run 3: new data arrives ---

source_batch_2 = [
    {"trade_id": "T007", "symbol": "EURUSD", "price": "1.0847", "volume": 600, "side": "BUY", "event_time": "2026-04-04T08:16:00Z", "source": "finnhub"},
    {"trade_id": "T008", "symbol": "USDJPY", "price": "145.15", "volume": 900, "side": "SELL", "event_time": "2026-04-04T08:16:01Z", "source": "finnhub"},
]

print(f"\n{'=' * 60}")
print("RUN 3: New Batch")
print("=" * 60)
report3 = pipeline.run(source_batch_2, "batch2-2026-04-04")
print(f"  Input: {report3['input_count']}")
print(f"  Ingested: {report3['ingested']}")
print(f"  Store total: {report3['metrics']['store_total']}")

# --- Final summary ---
print(f"\n{'=' * 60}")
print("FINAL SUMMARY")
print("=" * 60)
print(f"  Total records in store: {len(pipeline.store)}")
print(f"  Total quarantined: {len(pipeline.quarantine)}")
print(f"  Total seen keys: {len(pipeline.seen_keys)}")

print(f"\n  Quarantine details:")
for q in pipeline.quarantine:
    print(f"    [{q['stage']}] {q['error']}")
    print(f"      Record: {json.dumps(q['record'], default=str)[:80]}...")

print(f"\n  Lineage:")
for l in pipeline.lineage:
    print(f"    {l['run_id']}: {l['records_in']} in -> {l['records_out']} out "
          f"(quarantined: {l['quarantined']}, dupes: {l['duplicates']}, {l['duration_ms']}ms)")

# Try next:
# 1. Add a "reprocess quarantine" method that re-runs quarantined records through the pipeline.
# 2. Add a per-symbol volume metric to the report.
# 3. Implement the pipeline as a class with pluggable validation stages.

## 11. Exit Criteria

Do not move on until you can say yes to these without searching:

- I can implement retry logic with exponential backoff and jitter, and explain why jitter prevents thundering herd problems.
- I can implement a circuit breaker and explain the closed/open/half-open state transitions.
- I can explain when NOT to retry (non-idempotent operations, 4xx errors, poison pills).
- I can design an idempotency key from source, business key, and event time.
- I can explain the difference between exactly-once and at-least-once, and why at-least-once plus idempotency is the practical standard.
- I can define a data contract for a trade event and distinguish breaking from non-breaking schema changes.
- I can build a multi-stage quality gate that separates blocking errors from warnings.
- I can implement a quarantine pattern that preserves bad records with error context for later investigation.
- I can explain data lineage at the table level and design a safe backfill with idempotency and time boundaries.
- I can describe log-based, trigger-based, and timestamp-based CDC and explain the tradeoffs, especially around delete capture.
- I can make an informed batch vs stream decision based on latency requirements, debugging needs, and workload characteristics.
- I can define meaningful SLIs and SLOs for a data pipeline and explain error budgets.
- I can build a complete ingestion pipeline that combines validation, quarantine, dedup, lineage, and metrics.

## 12. References

Concepts and patterns in this notebook are drawn from established industry practices and the following resources:

- AWS Architecture Blog: Exponential Backoff and Jitter
  https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/
- Martin Fowler: Circuit Breaker Pattern
  https://martinfowler.com/bliki/CircuitBreaker.html
- Designing Data-Intensive Applications (Martin Kleppmann) -- chapters on reliability, stream processing, and batch processing
- Confluent Documentation: Schema Registry and Schema Evolution
  https://docs.confluent.io/platform/current/schema-registry/
- Google SRE Book: Service Level Objectives
  https://sre.google/sre-book/service-level-objectives/
- Debezium Documentation: Change Data Capture
  https://debezium.io/documentation/
- OpenLineage: Open Standard for Data Lineage
  https://openlineage.io/
- Jay Kreps: Questioning the Lambda Architecture
  https://www.oreilly.com/radar/questioning-the-lambda-architecture/
- Apache Kafka Documentation: Exactly-Once Semantics
  https://kafka.apache.org/documentation/
- Python Standard Library: `functools`, `dataclasses`, `datetime`, `decimal`, `uuid`, `random`, `time`, `json`
  https://docs.python.org/3/library/

## Interview Question Bank

Use these after you finish the notebook. Focus on failure modes and tradeoffs.

- When should you retry, and when should you not retry?
- Why is idempotency a foundation concept for reliable data platforms?
- What is the difference between a dead-letter queue and a retry queue?
- What is a data contract, and why is it better than informal assumptions?
- How do schema evolution and backward compatibility relate to producer-consumer safety?
- What is the real difference between batch and stream processing in system behavior?
- What are SLIs, SLOs, and error budgets in a data platform context?
- Why is "exactly once" often more a systems contract than a magical runtime property?
